## 协调器-工作者模式

In [53]:
import os

from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.constants import START
from langgraph.graph import StateGraph
from pydantic import BaseModel

load_dotenv()

True

In [54]:
llm = ChatDeepSeek(
    model="deepseek-chat",
    api_key=os.getenv("API_KEY"),
)

### 定义结构化输出

In [55]:
from typing import List, TypedDict
from pydantic import Field


class Section(BaseModel):
    name: str = Field(description="报告章节的名称")
    description: str = Field(description="本章节中涵盖的主要主题和概念的简要概述")


class Sections(BaseModel):
    sections: List[Section] = Field(description="报告的章节")

planner = llm.with_structured_output(Sections)

### 定义状态

In [56]:

import operator
from typing import Annotated

# 协调器的状态
class State(TypedDict):

    topic: str
    sections: List[Section]
    completed_sections: Annotated[list, operator.add]
    final_report: str

# 工作者状态
class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]

### 定义图中节点

In [57]:
def orchestrator(state: State):
    topic = state["topic"]
    report_sections = planner.invoke(
        [
            SystemMessage(content="请根据用户输入的主题生成报告计划。"),
            HumanMessage(content="这是报告主题:{topic}".format(topic=topic))
        ],

    )

    return {
        "sections": report_sections.sections
    }

def llm_call(state: WorkerState):
    section_name = state["section"].name

    section = llm.invoke(
        [
            SystemMessage(content='根据提供的章节的名称和描述编写报告章节，每个章节中不包含序言，使用markdown格式。200字以内'),
            HumanMessage(content=f'这是章节的名称: {section_name}')
        ]
    )

    return {
        "completed_sections": [section.content]
    }

def synthesizer(state: State):
    completed_sections = state["completed_sections"]
    completed_report_sections = "\n\n".join(completed_sections)

    return {
        "final_report": completed_report_sections
    }

### 定义节点和图

In [58]:
from langgraph.constants import END
from langgraph.types import Send


def assign_workers(state: State):
    # 动态工作创建
    return [Send("llm_call", {"section": s}) for s in state["sections"]]


graph_builder = StateGraph(State)
graph_builder.add_node("orchestrator", orchestrator)

graph_builder.add_node("llm_call", llm_call)

graph_builder.add_node("synthesizer", synthesizer)

graph_builder.add_edge(START, "orchestrator")
graph_builder.add_conditional_edges("orchestrator", assign_workers, ["llm_call"])

graph_builder.add_edge('llm_call', 'synthesizer')
graph_builder.add_edge('synthesizer', END)

graph = graph_builder.compile()


### 测试

In [59]:
result = graph.invoke({
    "topic": "创建关于LLM缩放定律的报告"
})
print(result['final_report'])

# 引言：LLM缩放定律概述

大型语言模型（LLM）的性能随着模型规模、数据量和计算资源的增加而显著提升，这一现象被称为缩放定律。研究表明，模型性能与这些关键因素之间存在可预测的幂律关系。理解缩放定律对于高效分配计算资源、预测模型能力边界以及指导未来模型开发至关重要。本章将概述缩放定律的核心概念、实证基础及其在LLM发展中的意义。

# 缩放定律的理论基础

缩放定律描述了系统性能随规模变化的规律。其核心在于识别关键参数与系统规模之间的幂律关系，通常表达为Y = kX^α。其中，Y是性能指标，X是规模参数，k是常数，α是缩放指数。

理论基础主要源于量纲分析和自相似性原理。在计算领域，它解释了为什么增加模型参数、数据量和计算量可以持续提升性能，并预测了性能提升的极限。不同α值揭示了性能随规模是线性增长、收益递减还是超越线性增长。

## 关键研究发现与经验规律

本研究通过分析，揭示了以下核心发现与规律：

1.  **核心驱动因素**：研究发现，[此处简述最主要的驱动因素，例如：用户参与度、技术创新或政策支持]是推动[研究主题]发展的最关键因素，其影响显著高于其他变量。
2.  **阶段性发展规律**：[研究主题]的发展呈现出明显的阶段性特征，大致经历了[简述阶段，如：萌芽期、快速增长期、平台整合期]三个阶段，每个阶段的主导模式和挑战各有不同。
3.  **普遍性关联**：数据分析表明，[某个关键指标A]与[另一个关键指标B]之间存在显著的正相关关系（相关系数达X），这一规律在不同样本群体中均得到验证。
4.  **差异化经验**：对比分析发现，[不同群体/场景，如：不同规模企业、不同地区]在实践路径和成效上存在显著差异。[例如：大型组织更依赖体系化建设，而中小组织则更注重灵活性与快速迭代]。

## 缩放定律的影响因素分析

缩放定律的有效性受多种因素影响。首先，**模型架构**是关键，Transformer架构因其可扩展性而成为主流。其次，**数据集质量与规模**至关重要，高质量、大规模且多样化的数据是实现有效缩放的基础。再者，**计算资源**（算力）是直接约束，通常遵循计算量、模型参数和数据量三者同步增长的规律。此外，**训练方法**（如优化器、学习率调度）的改进也能提升缩放效率。最后，**评估任务**的复杂性也会影响观察到的规律，某些能力可能在达到特定规